# FabricNET 130-image physical benchmark

This notebook is an interactive wrapper around `scripts/run_fabricnet_benchmark.py`, the canonical reproducibility runner. It runs the authoritative NumPy/Pillow pipeline on all 130 images, writes strict JSON/CSV artifacts, reports candidate and accepted metrics separately, computes specimen-cluster bootstrap intervals, and compares against the committed benchmark.

The notebook does not duplicate benchmark serialization or silently replace the canonical output schema.

In [ ]:
# The current production path uses NumPy FFT and Pillow; no PyTorch or OpenCV runtime is required.
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

REPO_ROOT = Path.cwd()
RUNNER = REPO_ROOT / "scripts" / "run_fabricnet_benchmark.py"
if not RUNNER.is_file():
    raise FileNotFoundError(
        "Run this notebook from the Warp-Weft repository root so scripts/run_fabricnet_benchmark.py is available."
    )

KAGGLE_DATA = Path("/kaggle/input/fabric-data/fabricnet")
LOCAL_DATA = REPO_ROOT / "data" / "fabricnet"
DATA_DIR = KAGGLE_DATA if KAGGLE_DATA.is_dir() else LOCAL_DATA
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else REPO_ROOT / "results" / "fabricnet_benchmark_rerun"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Runner: {RUNNER}")
print(f"FabricNET data: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# Run the same reproducible implementation used by CI.
command = [
    sys.executable,
    str(RUNNER),
    "--data-dir", str(DATA_DIR),
    "--output-dir", str(OUTPUT_DIR),
]
subprocess.run(command, check=True)

RESULT_JSON = OUTPUT_DIR / "fabricnet_benchmark_rerun.json"
with RESULT_JSON.open("r", encoding="utf-8") as handle:
    benchmark = json.load(handle)

if benchmark.get("total_samples") != 130 or len(benchmark.get("samples", [])) != 130:
    raise RuntimeError("Benchmark artifact is not a complete 130-image run.")

df_res = pd.DataFrame(benchmark["samples"])
print(f"Verified samples: {len(df_res)}/130")
print(f"Schema: {benchmark['schema_version']}")
print(f"JSON: {RESULT_JSON}")
print(f"CSV: {OUTPUT_DIR / 'fabricnet_benchmark_rerun.csv'}")

In [ ]:
# Report candidate estimates and accepted measurements separately.
aggregate = benchmark["aggregate_metrics"]
for prediction_type in ("candidate_estimates", "accepted_measurements"):
    print(f"\n=== {prediction_type.replace('_', ' ').title()} ===")
    for axis in ("warp", "weft"):
        metrics = aggregate[prediction_type][axis]
        print(
            f"{axis.title()}: n={metrics['n_predicted']}/{metrics['n_total']} "
            f"coverage={metrics['coverage_pct']}% rejection={metrics['rejection_rate_pct']}% "
            f"MAE={metrics['mae']} RMSE={metrics['rmse']} "
            f"median_abs={metrics['median_absolute_error']} "
            f"bias={metrics['mean_signed_bias']} "
            f"within±2={metrics['within_2_pct']}%"
        )
        print(f"  MAE group-bootstrap 95% CI: {metrics['mae_bootstrap_95_ci']}")

print("\n=== Decision status ===")
print(aggregate["decision_status_breakdown"])
print("\n=== Per-weave accepted coverage ===")
for weave, metrics in aggregate["per_weave"].items():
    print(weave, {
        "warp": metrics["accepted_measurements"]["warp"]["coverage_pct"],
        "weft": metrics["accepted_measurements"]["weft"]["coverage_pct"],
    })

print("\n=== Comparison to committed benchmark ===")
print(benchmark["comparison_to_committed_benchmark"])

In [ ]:
# Six-panel diagnostic summary; all plotted predictions are explicitly candidate estimates.
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
colors = {"plain": "#1f77b4", "twill": "#2ca02c", "satin": "#d62728"}

for weave, group in df_res.groupby("texture"):
    axes[0, 0].scatter(
        group["gt_warp"], group["pred_warp"],
        label=f"{weave} (n={len(group)})",
        color=colors.get(weave, "gray"), alpha=0.7, s=35
    )
lims = [
    min(df_res["gt_warp"].min(), df_res["pred_warp"].min()) - 2,
    max(df_res["gt_warp"].max(), df_res["pred_warp"].max()) + 2,
]
axes[0, 0].plot(lims, lims, "k--", alpha=0.6)
axes[0, 0].set(title="Warp candidate: ground truth vs estimate", xlabel="Manual ends/cm", ylabel="Candidate ends/cm")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.25)

for weave, group in df_res.groupby("texture"):
    axes[0, 1].scatter(
        group["gt_weft"], group["pred_weft"],
        label=weave, color=colors.get(weave, "gray"), alpha=0.7, s=35
    )
lims = [
    min(df_res["gt_weft"].min(), df_res["pred_weft"].min()) - 2,
    max(df_res["gt_weft"].max(), df_res["pred_weft"].max()) + 2,
]
axes[0, 1].plot(lims, lims, "k--", alpha=0.6)
axes[0, 1].set(title="Weft candidate: ground truth vs estimate", xlabel="Manual picks/cm", ylabel="Candidate picks/cm")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.25)

error_data = [
    df_res.loc[df_res["texture"] == weave, "err_warp_candidate"].dropna().values
    for weave in ("plain", "twill", "satin")
]
axes[0, 2].boxplot(error_data, tick_labels=["Plain", "Twill", "Satin"])
axes[0, 2].set(title="Warp candidate absolute error", ylabel="Absolute error (ends/cm)")
axes[0, 2].grid(axis="y", alpha=0.25)

for weave, group in df_res.groupby("texture"):
    axes[1, 0].hist(
        group["signal_quality_index"].dropna(), bins=15, alpha=0.5,
        label=weave, color=colors.get(weave, "gray")
    )
axes[1, 0].set(title="Signal Quality Index (heuristic)", xlabel="SQI", ylabel="Samples")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.25)

for axis, key in ((axes[1, 1], "accepted_warp"), (axes[1, 2], "accepted_weft")):
    accepted_count = int(df_res[key].notna().sum())
    axis.bar(["Accepted", "Rejected"], [accepted_count, len(df_res) - accepted_count])
    axis.set_title(f"{key.replace('_', ' ').title()} (n={accepted_count}/130)")
    axis.set_ylabel("Samples")
    axis.grid(axis="y", alpha=0.25)

plt.tight_layout()
plot_path = OUTPUT_DIR / "fabricnet_benchmark_report.png"
plt.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved diagnostic plot: {plot_path}")

In [ ]:
# Inspect the two historical exemplar samples without changing the benchmark.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for axis, image_name in zip(axes, ("W012.jpg", "W001.jpg")):
    image_path = DATA_DIR / image_name
    with Image.open(image_path) as image:
        axis.imshow(image.convert("RGB"))
    sample = df_res.loc[df_res["image_id"] == image_name].iloc[0]
    axis.set_title(
        f"{image_name} | GT={sample['gt_warp']:.0f}×{sample['gt_weft']:.0f} "
        f"| Candidate={sample['pred_warp']:.1f}×{sample['pred_weft']:.1f}\n"
        f"Status={sample['decision_status']} | accepted warp={sample['accepted_warp']} "
        f"| accepted weft={sample['accepted_weft']}"
    )
    axis.axis("off")
plt.tight_layout()
plt.show()

## Interpretation guardrails

- Candidate-estimate errors include FFT fallback candidates when fusion rejects an axis. They are not accepted-measurement accuracy.
- Accepted-only error statistics must always be read with coverage and rejection rates.
- The orientation-invariant paired score is ground-truth-assisted and offline only.
- FabricNET's nominal 5 × 4 mm field of view gives 1280 × 1200 px/cm, but it has not been verified with a stage micrometer.
- The reported calibration envelope is a configured ±5% assumption, not a measured per-device uncertainty.
- Exact reproduction demonstrates deterministic code/data consistency, not satisfaction of the client's ±2 tolerance in a smartphone/factory acquisition domain.